# Fit By Form — FASHN VTON 1.5 one-click GPU smoke test
This notebook uses the official FASHN sample pair only. It does not upload customer photos. Select **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. GPU availability on free Colab is not guaranteed.

In [6]:
import os, subprocess, sys, json, pathlib
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU attached. In Colab choose Runtime > Change runtime type > T4 GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
VRAM GB: 14.56


In [7]:
import os, subprocess, pathlib
WORK = pathlib.Path('/content/fit-by-form-vton-test')
if WORK.exists():
    subprocess.run(['rm','-rf',str(WORK)], check=True)
WORK.mkdir(parents=True)
os.chdir(WORK)
UPSTREAM_COMMIT = '7c0f10af3f91ad4048fe9729c470a13ef905d25a'
subprocess.run(['git','clone','https://github.com/fashn-AI/fashn-vton-1.5.git','fashn-vton-1.5'], check=True)
os.chdir(WORK/'fashn-vton-1.5')
subprocess.run(['git','checkout',UPSTREAM_COMMIT], check=True)
base = 'https://raw.githubusercontent.com/flowrian2026-max/Fit/main/experiments/virtual-tryon/'
for name in ['apply_fashn_parser_free_overlay.py','verify_fashn_parser_free.py','run_fashn_flatlay_smoke.py']:
    subprocess.run(['curl','-fsSL',base+name,'-o',name], check=True)
subprocess.run([sys.executable,'apply_fashn_parser_free_overlay.py','.'], check=True)
subprocess.run([sys.executable,'verify_fashn_parser_free.py','.'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'verify_fashn_parser_free.py', '.'], returncode=0)

In [8]:
# Install the patched source and download the ~2 GB official model weights.
subprocess.run([sys.executable,'-m','pip','install','-e','.'], check=True)
subprocess.run([sys.executable,'scripts/download_weights.py','--weights-dir','./weights'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/download_weights.py', '--weights-dir', './weights'], returncode=0)

In [9]:
# Download the two public example images from the pinned upstream repository.
import urllib.request
data = pathlib.Path('examples/data')
data.mkdir(parents=True, exist_ok=True)
raw = f'https://raw.githubusercontent.com/fashn-AI/fashn-vton-1.5/{UPSTREAM_COMMIT}/examples/data/'
for name in ['model.webp','garment.webp']:
    urllib.request.urlretrieve(raw+name, data/name)
print('Downloaded official sample inputs:', list(p.name for p in data.iterdir()))

Downloaded official sample inputs: ['garment.webp', 'model.webp']


In [ ]:
# Run one parser-free flat-lay test and record timing + peak VRAM.
subprocess.run([
    sys.executable, 'run_fashn_flatlay_smoke.py',
    '--weights-dir','./weights',
    '--person-image','./examples/data/model.webp',
    '--garment-image','./examples/data/garment.webp',
    '--category','tops', '--steps','30', '--require-cuda',
    '--output','./tryon-output.png', '--metrics','./tryon-metrics.json'
], check=True)
from IPython.display import display, Image
display(Image(filename='tryon-output.png'))
print(pathlib.Path('tryon-metrics.json').read_text())

## Important
- This is a technical smoke test, not proof of production quality.
- The first run downloads roughly 2 GB of model weights and may take time.
- Do not upload customer photos to this notebook.
- Save the output and metrics outside the temporary Colab runtime if you need to keep them.